# 语境
情境工程是一种构建动态系统的实践，该系统以正确的格式提供正确的信息和工具，以便AI应用程序能够完成任务。情境可以从两个关键维度来描述：

- 根据可变性：
    - 静态上下文：执行期间不会改变的不可变数据（例如，用户元数据、数据库连接、工具）
    - 动态上下文：随着应用程序运行而变化的可变数据（例如，对话历史记录、中间结果、工具调用观察）
- 按寿命：
    - 运行时上下文：单次运行或调用范围内的数据
    - 跨对话上下文：跨多个对话或会话持续的数据


LangGraph 提供了三种管理上下文的方式，结合了可变性和生命周期维度：

| 上下文类型 | 描述 | 可变性 | 寿命 | 访问方法 |
| :--- | :--- | :--- | :--- | :--- |
| 静态运行时上下文 | 启动时传递的用户元数据、工具、数据库连接 | 静止的 | 单次运行 | context 参数为 invoke／stream |
| 动态运行时上下文 （状态） | 单次运行期间不断变化的可变数据 | 动态的 | 单次运行 | LangGraph 状态对象 |
| 动态跨对话上下文 （商店） | 跨对话共享的持久数据 | 动态的 | 跨对话 | LangGraph 商店 |


## 静态运行时上下文

静态运行时上下文表示不可更改的数据，如用户元数据、工具和数据库连接，这些数据在运行开始时通过 invoke/stream 的上下文参数传递给应用程序。这些数据在执行过程中不会改变。

In [ ]:
@dataclass
class ContextSchema:
    user_name: str

graph.invoke( 
    {"messages": [{"role": "user", "content": "hi!"}]}, 
    context={"user_name": "John Smith"} 
)

In [ ]:
from langchain_core.messages import AnyMessage
from langgraph.runtime import get_runtime
from langgraph.prebuilt.chat_agent_executor import AgentState
from langgraph.prebuilt import create_react_agent

def prompt(state: AgentState) -> list[AnyMessage]:
    runtime = get_runtime(ContextSchema)
    system_msg = f"You are a helpful assistant. Address the user as {runtime.context.user_name}."
    return [{"role": "system", "content": system_msg}] + state["messages"]

agent = create_react_agent(
    model="anthropic:claude-3-7-sonnet-latest",
    tools=[get_weather],
    prompt=prompt,
    context_schema=ContextSchema
)

agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]},
    context={"user_name": "John Smith"}
)

In [ ]:
from langgraph.runtime import Runtime

def node(state: State, config: Runtime[ContextSchema]):
    user_name = runtime.context.user_name
    ...

In [ ]:
from langgraph.runtime import get_runtime

@tool
def get_user_email() -> str:
    """Retrieve user information based on user ID."""
    # simulate fetching user info from a database
    runtime = get_runtime(ContextSchema)
    email = get_user_email_from_db(runtime.context.user_name)
    return email

详情请参见[工具调用](https://langchain-ai.github.io/langgraph/how-tos/tool-calling/#configuration)指南。

## 动态运行时上下文（状态）¶
动态运行时上下文表示在单次运行期间可能发生变化的可变数据，并通过 LangGraph 状态对象进行管理。这些数据包括对话历史记录、中间结果以及来自工具或 LLM 输出的值。在 LangGraph 中，状态对象在运行期间充当短期记忆。

示例显示如何将状态合并到代理提示中。

状态也可以通过代理的工具访问，这些工具可以根据需要读取或更新状态。有关详细信息，请参阅工具调用指南。

In [ ]:
from langchain_core.messages import AnyMessage
from langchain_core.runnables import RunnableConfig
from langgraph.prebuilt import create_react_agent
from langgraph.prebuilt.chat_agent_executor import AgentState

class CustomState(AgentState): 
    user_name: str

def prompt(
    state: CustomState
) -> list[AnyMessage]:
    user_name = state["user_name"]
    system_msg = f"You are a helpful assistant. User's name is {user_name}"
    return [{"role": "system", "content": system_msg}] + state["messages"]

agent = create_react_agent(
    model="anthropic:claude-3-7-sonnet-latest",
    tools=[...],
    state_schema=CustomState, 
    prompt=prompt
)

agent.invoke({
    "messages": "hi!",
    "user_name": "John Smith"
})

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph

class CustomState(TypedDict): 
    messages: list[AnyMessage]
    extra_field: int

def node(state: CustomState): 
    messages = state["messages"]
    ...
    return { 
        "extra_field": state["extra_field"] + 1
    }

builder = StateGraph(State)
builder.add_node(node)
builder.set_entry_point("node")
graph = builder.compile()

## 动态跨对话上下文（商店）¶
动态跨对话上下文表示跨多个对话或会话的持久可变数据，并通过 LangGraph 存储进行管理。这些数据包括用户个人资料、偏好设置和历史交互。LangGraph 存储充当跨多次运行的长期内存。这可用于读取或更新持久性事实（例如，用户个人资料、偏好设置、先前交互）。

有关详细信息，请参阅内存指南。